In [1]:
!pip -q install -U sentence-transformers pandas scikit-learn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.6/740.6 kB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 78.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 34.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.0.6 which is incompatible.
dask-cudf-cu13 26.6.0 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.6 which is incompatible.
cudf-cu13 26.6.0 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.6 which is incompatible.


In [3]:
%pip install -q rank-bm25

In [4]:
import os
import re
import json
import numpy as np
import pandas as pd
import torch

from datetime import datetime
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from rank_bm25 import BM25Okapi
from IPython.display import display

In [7]:
queries = pd.read_csv("queries_merged.csv", encoding="utf-8-sig")
corpus = pd.read_csv("corpus_merged.csv", encoding="utf-8-sig")
relevance = pd.read_csv("relevance_merged.csv", encoding="utf-8-sig")

print("Queries:", len(queries))
print("Passages:", len(corpus))
print("Relevance judgments:", len(relevance))

print("\nQuery columns:", queries.columns.tolist())
print("Corpus columns:", corpus.columns.tolist())
print("Relevance columns:", relevance.columns.tolist())

Queries: 420
Passages: 398
Relevance judgments: 420

Query columns: ['query_id', 'task_type', 'language', 'domain', 'query_text_ml', 'query_type', 'query_topic_ml', 'expected_information_ml', 'gold_passage_ids', 'gold_relevance_levels', 'difficulty', 'source_title', 'source_chapter', 'source_printed_pages', 'source_pdf_pages', 'source_evidence_status', 'query_generation_method', 'validation_status', 'reviewer_notes', 'split']
Corpus columns: ['passage_id', 'source_id', 'source_title', 'passage_text_ml', 'chapter_title', 'printed_page_start', 'printed_page_end', 'pdf_page_start', 'pdf_page_end', 'passage_topic_ml', 'passage_content_type', 'source_text_status', 'text_normalization', 'related_query_ids', 'corpus_split', 'review_notes']
Relevance columns: ['query_id', 'passage_id', 'relevance_grade', 'relevance_rationale_ml', 'evidence_type', 'evidence_status', 'validation_status', 'reviewer_notes']


In [9]:
required_queries = {"query_id", "query_text_ml"}
required_corpus = {"passage_id", "passage_text_ml"}
required_relevance = {"query_id", "passage_id", "relevance_grade"}

assert required_queries.issubset(queries.columns), "Missing query columns"
assert required_corpus.issubset(corpus.columns), "Missing corpus columns"
assert required_relevance.issubset(relevance.columns), "Missing relevance columns"

queries = queries.dropna(
    subset=["query_id", "query_text_ml"]
).copy()

corpus = corpus.dropna(
    subset=["passage_id", "passage_text_ml"]
).copy()

relevance = relevance.dropna(
    subset=["query_id", "passage_id", "relevance_grade"]
).copy()

for df, columns in [
    (queries, ["query_id"]),
    (corpus, ["passage_id"]),
    (relevance, ["query_id", "passage_id"])
]:
    for col in columns:
        df[col] = df[col].astype(str).str.strip()

relevance["relevance_grade"] = pd.to_numeric(
    relevance["relevance_grade"], errors="coerce"
)

relevance = relevance.dropna(subset=["relevance_grade"]).copy()
relevance["relevance_grade"] = relevance["relevance_grade"].astype(int)

relevance = relevance[
    relevance["relevance_grade"].isin([0, 1, 2])
].copy()

print("Duplicate query IDs:", queries["query_id"].duplicated().sum())
print("Duplicate passage IDs:", corpus["passage_id"].duplicated().sum())

if queries["query_id"].duplicated().any():
    raise ValueError("Please fix duplicate query IDs.")

if corpus["passage_id"].duplicated().any():
    raise ValueError("Please fix duplicate passage IDs.")

relevance = relevance.drop_duplicates(
    ["query_id", "passage_id"], keep="last"
)

valid_queries = set(queries["query_id"])
valid_passages = set(corpus["passage_id"])

relevance = relevance[
    relevance["query_id"].isin(valid_queries)
    & relevance["passage_id"].isin(valid_passages)
].copy()

query_ids = queries["query_id"].tolist()
passage_ids = corpus["passage_id"].tolist()

query_texts = queries["query_text_ml"].astype(str).tolist()
passage_texts = corpus["passage_text_ml"].astype(str).tolist()

print("Prepared successfully.")

Duplicate query IDs: 0
Duplicate passage IDs: 0
Prepared successfully.


In [10]:
MODEL_NAME = "Qwen/Qwen3-Embedding-0.6B"

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", device)

model = SentenceTransformer(
    MODEL_NAME,
    device=device,
    trust_remote_code=True
)

print("Model loaded:", MODEL_NAME)
print("Maximum sequence length:", model.max_seq_length)

Device: cuda


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/17.2k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

...

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

config.json:   0%|          | 0.00/313 [00:00<?, ?B/s]

Model loaded: Qwen/Qwen3-Embedding-0.6B
Maximum sequence length: 32768


In [11]:
query_embeddings = model.encode(
    query_texts,
    batch_size=8,
    normalize_embeddings=True,
    convert_to_numpy=True,
    show_progress_bar=True
)

passage_embeddings = model.encode(
    passage_texts,
    batch_size=8,
    normalize_embeddings=True,
    convert_to_numpy=True,
    show_progress_bar=True
)

print("Query embeddings:", query_embeddings.shape)
print("Passage embeddings:", passage_embeddings.shape)

Batches:   0%|          | 0/53 [00:00<?, ?it/s]

Batches:   0%|          | 0/50 [00:00<?, ?it/s]

Query embeddings: (420, 1024)
Passage embeddings: (398, 1024)


In [12]:
# 1. Dense retrieval using Qwen3
dense_scores = query_embeddings @ passage_embeddings.T

# 2. Character-level TF-IDF
tfidf_vectorizer = TfidfVectorizer(
    analyzer="char",
    ngram_range=(2, 5),
    sublinear_tf=True,
    min_df=1,
    max_features=300000
)

passage_tfidf = tfidf_vectorizer.fit_transform(passage_texts)
query_tfidf = tfidf_vectorizer.transform(query_texts)

tfidf_scores = cosine_similarity(query_tfidf, passage_tfidf)

# 3. BM25
def tokenize(text):
    return re.findall(r"\w+", str(text).lower())

passage_tokens = [tokenize(text) for text in passage_texts]
query_tokens = [tokenize(text) for text in query_texts]

bm25 = BM25Okapi(passage_tokens)

bm25_scores = np.array([
    bm25.get_scores(tokens)
    for tokens in query_tokens
])

# 4. Hybrid: 50% Dense + 50% TF-IDF
def normalize_rows(scores):
    minimum = scores.min(axis=1, keepdims=True)
    maximum = scores.max(axis=1, keepdims=True)

    return (scores - minimum) / np.maximum(
        maximum - minimum, 1e-12
    )

hybrid_scores = (
    normalize_rows(dense_scores)
    + normalize_rows(tfidf_scores)
) / 2

score_sets = {
    "Dense": dense_scores,
    "Hybrid": hybrid_scores,
    "TF-IDF": tfidf_scores,
    "BM25": bm25_scores
}

print("All retrieval scores computed.")

All retrieval scores computed.


In [13]:
gold = {
    (row.query_id, row.passage_id): int(row.relevance_grade)
    for row in relevance.itertuples(index=False)
}

print("Gold judgments:", len(gold))
print("Grade 2 judgments:", (relevance["relevance_grade"] == 2).sum())
print("Grade 1 judgments:", (relevance["relevance_grade"] == 1).sum())
print("Grade 0 judgments:", (relevance["relevance_grade"] == 0).sum())

Gold judgments: 420
Grade 2 judgments: 420
Grade 1 judgments: 0
Grade 0 judgments: 0


In [14]:
def dcg(grades, k):
    grades = np.asarray(grades[:k], dtype=float)

    if len(grades) == 0:
        return 0.0

    gains = (2 ** grades - 1) / np.log2(
        np.arange(2, len(grades) + 2)
    )

    return float(gains.sum())


def evaluate_method(scores, method_name):
    rows = []
    ranking_rows = []

    for i, qid in enumerate(query_ids):
        order = np.argsort(-scores[i])

        ranked_ids = [passage_ids[j] for j in order]

        grades = [
            gold.get((qid, pid), 0)
            for pid in ranked_ids
        ]

        relevant_ids = {
            pid for (gold_qid, pid), grade in gold.items()
            if gold_qid == qid and grade == 2
        }

        row = {
            "Model": method_name,
            "query_id": qid
        }

        for k in [1, 3, 5, 10]:
            row[f"Recall@{k}"] = float(
                bool(relevant_ids.intersection(ranked_ids[:k]))
            )

        first_rank = next(
            (
                rank for rank, grade in enumerate(grades, start=1)
                if grade == 2
            ),
            None
        )

        row["MRR@10"] = (
            1 / first_rank
            if first_rank is not None and first_rank <= 10
            else 0.0
        )

        # Ideal DCG based on the available relevance judgments
        judged_grades = [
            grade
            for (gold_qid, _), grade in gold.items()
            if gold_qid == qid
        ]

        ideal_grades = sorted(judged_grades, reverse=True)[:10]
        ideal_score = dcg(ideal_grades, 10)

        row["nDCG@10"] = (
            dcg(grades, 10) / ideal_score
            if ideal_score > 0 else np.nan
        )

        rows.append(row)

        # Save the ranked passages for later inspection
        for rank, j in enumerate(order, start=1):
            pid = passage_ids[j]

            ranking_rows.append({
                "model": method_name,
                "query_id": qid,
                "query_text_ml": query_texts[i],
                "rank": rank,
                "passage_id": pid,
                "passage_text_ml": passage_texts[j],
                "similarity_score": float(scores[i, j]),
                "relevance_grade": gold.get((qid, pid), "")
            })

    return pd.DataFrame(rows), pd.DataFrame(ranking_rows)


metric_frames = []
ranking_frames = []

for name, scores in score_sets.items():
    per_query, ranked = evaluate_method(scores, name)

    metric_frames.append(per_query)
    ranking_frames.append(ranked)

all_metrics = pd.concat(metric_frames, ignore_index=True)
all_rankings = pd.concat(ranking_frames, ignore_index=True)

print("Evaluation completed.")

Evaluation completed.


In [15]:
metric_columns = [
    "Recall@1",
    "Recall@3",
    "Recall@5",
    "Recall@10",
    "MRR@10",
    "nDCG@10"
]

summary = (
    all_metrics.groupby("Model")[metric_columns]
    .mean()
    .mul(100)
    .round(2)
    .reset_index()
)

print(
    f"Model: {MODEL_NAME} | "
    f"{len(queries)} queries | {len(corpus)} passages"
)

display(
    summary.style
    .format({col: "{:.1f}%" for col in metric_columns})
    .highlight_max(
        subset=metric_columns,
        axis=0,
        props="color: #34d399; font-weight: bold;"
    )
)

Model: Qwen/Qwen3-Embedding-0.6B | 420 queries | 398 passages


,Model,Recall@1,Recall@3,Recall@5,Recall@10,MRR@10,nDCG@10
0,BM25,31.4%,44.0%,51.4%,60.0%,40.0%,44.7%
1,Dense,44.5%,61.2%,67.9%,75.5%,54.7%,59.7%
2,Hybrid,71.4%,87.9%,91.4%,95.7%,80.2%,84.0%
3,TF-IDF,64.8%,81.7%,86.2%,92.9%,74.1%,78.6%


In [16]:
import json
import numpy as np
from datetime import datetime

def bootstrap_ci(values, n_bootstrap=2000, seed=42):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]

    if len(values) == 0:
        return [None, None, None]

    rng = np.random.default_rng(seed)
    means = [
        rng.choice(values, size=len(values), replace=True).mean()
        for _ in range(n_bootstrap)
    ]

    return [
        round(float(values.mean()), 6),
        round(float(np.percentile(means, 2.5)), 6),
        round(float(np.percentile(means, 97.5)), 6)
    ]


metric_map = {
    "recall@1": "Recall@1",
    "recall@3": "Recall@3",
    "recall@5": "Recall@5",
    "recall@10": "Recall@10",
    "mrr@10": "MRR@10",
    "ndcg@10": "nDCG@10"
}

variant_names = {
    "Dense": "dense",
    "Hybrid": "hybrid",
    "TF-IDF": "tfidf_char",
    "BM25": "bm25"
}

variants = {}

for method, variant_name in variant_names.items():
    subset = all_metrics[all_metrics["Model"] == method]

    if subset.empty:
        continue

    variants[variant_name] = {
        metric_name: bootstrap_ci(subset[column].values)
        for metric_name, column in metric_map.items()
    }

result = {
    "model": None,
    "embed_model": "Qwen/Qwen3-Embedding-0.6B",
    "few_shot_k": 0,
    "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "passage_retrieval": {
        "n_queries": int(queries["query_id"].nunique()),
        "n_passages": int(corpus["passage_id"].nunique()),
        "primary": "dense",
        "embed_model": "Qwen/Qwen3-Embedding-0.6B",
        "max_seq_length": int(model.max_seq_length),
        "truncated_passages": None,
        "queries_referring_to_page": None,
        "variants": variants
    }
}

filename = "qwen3-embedding-results.json"

with open(filename, "w", encoding="utf-8") as f:
    json.dump(result, f, ensure_ascii=False, indent=2, allow_nan=False)

print("JSON file saved:", filename)
print(json.dumps(result, ensure_ascii=False, indent=2))

JSON file saved: qwen3-embedding-results.json
{
  "model": null,
  "embed_model": "Qwen/Qwen3-Embedding-0.6B",
  "few_shot_k": 0,
  "timestamp": "2026-10-12 05:30:23",
  "passage_retrieval": {
    "n_queries": 420,
    "n_passages": 398,
    "primary": "dense",
    "embed_model": "Qwen/Qwen3-Embedding-0.6B",
    "max_seq_length": 32768,
    "truncated_passages": null,
    "queries_referring_to_page": null,
    "variants": {
      "dense": {
        "recall@1": [
          0.445238,
          0.4,
          0.490536
        ],
        "recall@3": [
          0.611905,
          0.566667,
          0.657143
        ],
        "recall@5": [
          0.678571,
          0.635655,
          0.72381
        ],
        "recall@10": [
          0.754762,
          0.714286,
          0.795238
        ],
        "mrr@10": [
          0.547406,
          0.507047,
          0.588108
        ],
        "ndcg@10": [
          0.597271,
          0.558687,
          0.636933
        ]
      },
   

In [17]:
from google.colab import files

files.download("qwen3-embedding-results.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>